# Stock Markets Analytics Zoomcamp — Module 1 Homework Solutions (2026 cohort)

**Author:** Homework solutions
**Date:** 2026-09-14

This notebook contains complete working code for Questions 1–4 of Module 1 Homework:
1. **Q1** — S&P 500 additions by year (Wikipedia)
2. **Q2** — World Indices YTD performance vs S&P 500 (Yahoo Finance)
3. **Q3** — S&P 500 correction/drawdown analysis since 1950 (Yahoo Finance)
4. **Q4** — Amazon (AMZN) earnings surprise analysis (Yahoo Finance)

Free-text questions (Q5, Q6) are answered in README.md.


In [1]:
# Imports and setup
import warnings
warnings.filterwarnings('ignore')

import requests
import pandas as pd
import numpy as np
import yfinance as yf
from io import StringIO
from datetime import datetime

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully.")
print(f"yfinance version: {yf.__version__}")
print(f"pandas version:  {pd.__version__}")
print(f"numpy version:   {np.__version__}")


Libraries imported successfully.
yfinance version: 1.7.0
pandas version:  2.3.1
numpy version:   1.24.3


---
## Question 1: S&P 500 Stocks Added to the Index

> Which (full) year had the highest number of additions, starting from 2020?
>
> Additional: How many current S&P 500 stocks have been in the index for more than 20 years?


In [2]:
# Scrape the Wikipedia list of S&P 500 companies
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))

# First table = current constituents
sp500 = tables[0].copy()
print(f"Constituents table shape: {sp500.shape}")
print("Columns:", list(sp500.columns))
sp500.head()


Constituents table shape: (503, 8)
Columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [3]:
# Parse 'Date added' and extract year
sp500['Date added'] = pd.to_datetime(sp500['Date added'], errors='coerce')
sp500['Year added'] = sp500['Date added'].dt.year

# Count additions per year
additions_by_year = sp500.groupby('Year added').size().sort_index()
print("=== Additions by year (all years) ===")
print(additions_by_year.to_string())


=== Additions by year (all years) ===
Year added
1957    52
1964     1
1965     2
1969     2
1970     2
1972     2
1973     2
1974     1
1975     2
1976    11
1978     1
1979     1
1980     3
1981     3
1982     5
1983     2
1984     4
1985     7
1986     3
1987     2
1988     4
1989     4
1991     1
1992     2
1993     3
1994     6
1995     7
1996     2
1997    14
1998    11
1999     9
2000     9
2001     8
2002    12
2003     5
2004     6
2005     7
2006     9
2007    11
2008    16
2009    12
2010     8
2011    10
2012    14
2013     9
2014     8
2015    14
2016    21
2017    22
2018    13
2019    21
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13


In [4]:
# Focus on 2020+; exclude 2026 since it's a partial year
recent_full = additions_by_year[(additions_by_year.index >= 2020) & (additions_by_year.index <= 2025)]
print("=== Additions by year (2020-2025, full years only) ===")
print(recent_full.to_string())

max_year = int(recent_full.idxmax())
max_count = int(recent_full.max())
print(f"\n>>> Q1 ANSWER: Year with most additions (full years 2020-2025): {max_year} with {max_count} additions")


=== Additions by year (2020-2025, full years only) ===
Year added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18

>>> Q1 ANSWER: Year with most additions (full years 2020-2025): 2025 with 18 additions


In [5]:
# Additional: stocks in index for more than 20 years (as of 2026-09-14)
cutoff_date = pd.Timestamp('2006-09-14')  # exactly 20 years ago
more_than_20 = sp500[sp500['Date added'] < cutoff_date]
print(f">>> Additional: Stocks in index > 20 years (added before {cutoff_date.date()}): {len(more_than_20)}")

# Alternatives
print(f"    Alternative (added in or before 2005): {len(sp500[sp500['Year added'] <= 2005])}")
print(f"    Alternative (added in or before 2004): {len(sp500[sp500['Year added'] <= 2004])}")


>>> Additional: Stocks in index > 20 years (added before 2006-09-14): 224
    Alternative (added in or before 2005): 218
    Alternative (added in or before 2004): 211


---
## Question 2: World Indices YTD (as of 21 August 2026)

> How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?
>
> Additional: Compare over 3, 5, and 10 year windows.


In [6]:
# Define the 11 indices (10 countries + US benchmark)
indices = {
    '^GSPC':     'United States - S&P 500',
    '000001.SS': 'China - Shanghai Composite',
    '^HSI':      'Hong Kong - Hang Seng',
    '^AXJO':     'Australia - S&P/ASX 200',
    '^NSEI':     'India - Nifty 50',
    '^GSPTSE':   'Canada - S&P/TSX Composite',
    '^GDAXI':    'Germany - DAX',
    '^FTSE':     'UK - FTSE 100',
    '^N225':     'Japan - Nikkei 225',
    '^MXX':      'Mexico - IPC Mexico',
    '^BVSP':     'Brazil - Ibovespa',
}

start_date = '2026-01-01'
end_date   = '2026-08-22'  # +1 day because yfinance end is exclusive

# Download daily prices
print("Downloading YTD data...")
data = yf.download(list(indices.keys()), start=start_date, end=end_date,
                   auto_adjust=False, progress=False)
close = data['Close']
print(f"Shape: {close.shape}; dates: {close.index.min().date()} to {close.index.max().date()}")


Shape: (167, 11); dates: 2026-01-01 to 2026-08-21


In [7]:
# Per-index first/last valid trading day return (handles market-holiday NaNs)
ytd_results = {}
for t in indices:
    s = close[t].dropna()
    if len(s) == 0:
        ytd_results[t] = (np.nan, None, None)
        continue
    first_p, last_p = s.iloc[0], s.iloc[-1]
    ytd_results[t] = ((last_p / first_p - 1) * 100, s.index[0].date(), s.index[-1].date())

sp500_ret = ytd_results['^GSPC'][0]

print(f"{'Index':40s} {'Ticker':10s} {'YTD Return':>10s}   Window")
print("-" * 90)
for t in sorted(indices, key=lambda x: -(ytd_results[x][0] if not np.isnan(ytd_results[x][0]) else -999)):
    ret, fd, ld = ytd_results[t]
    flag = " <-- BETTER than S&P 500" if t != '^GSPC' and ret > sp500_ret else ""
    print(f"{indices[t]:40s} {t:10s} {ret:+9.2f}%    {fd} -> {ld}{flag}")

better_count = sum(1 for t in indices if t != '^GSPC' and ytd_results[t][0] > sp500_ret)
print(f"\n>>> Q2 ANSWER: Indices with better YTD than S&P 500: {better_count} out of 10")
print(f"    S&P 500 YTD: {sp500_ret:+.2f}%")


Index                                    Ticker     YTD Return   Window
------------------------------------------------------------------------------------------
Japan - Nikkei 225                       ^N225         +27.36%    2026-01-05 -> 2026-08-21 <-- BETTER than S&P 500
Canada - S&P/TSX Composite               ^GSPTSE       +14.86%    2026-01-02 -> 2026-08-21 <-- BETTER than S&P 500
United States - S&P 500                  ^GSPC         +11.90%    2026-01-02 -> 2026-08-21
UK - FTSE 100                            ^FTSE          +8.70%    2026-01-02 -> 2026-08-21
Brazil - Ibovespa                        ^BVSP          +6.54%    2026-01-02 -> 2026-08-21
Germany - DAX                            ^GDAXI         +6.51%    2026-01-02 -> 2026-08-21
Australia - S&P/ASX 200                  ^AXJO          +3.79%    2026-01-02 -> 2026-08-21
Mexico - IPC Mexico                      ^MXX           +2.48%    2026-01-02 -> 2026-08-21
Hong Kong - Hang Seng                    ^HSI           -1.25

In [8]:
# Additional: 3-, 5-, and 10-year returns
for label, start_d in [('3y', '2023-08-21'), ('5y', '2021-08-21'), ('10y', '2016-08-21')]:
    d = yf.download(list(indices.keys()), start=start_d, end='2026-08-22',
                    auto_adjust=False, progress=False)
    c = d['Close']
    rets = {}
    for t in indices:
        s = c[t].dropna()
        rets[t] = (s.iloc[-1] / s.iloc[0] - 1) * 100 if len(s) >= 2 else np.nan
    sp = rets['^GSPC']
    bn = sum(1 for t in indices if t != '^GSPC' and not np.isnan(rets[t]) and rets[t] > sp)
    print(f"\n--- {label} (since {start_d})  | S&P 500 = {sp:+.2f}%  | Beating it: {bn}/10 ---")
    for t in sorted(indices, key=lambda x: -(rets[x] if not np.isnan(rets[x]) else -999)):
        b = " <-- better" if t != '^GSPC' and rets[t] > sp else ""
        print(f"  {indices[t]:40s}: {rets[t]:+7.2f}%{b}")



--- 3y (since 2023-08-21)  | S&P 500 = +74.43%  | Beating it: 2/10 ---
  Japan - Nikkei 225                      : +109.14% <-- better
  Canada - S&P/TSX Composite              :  +85.09% <-- better
  United States - S&P 500                 :  +74.43%
  Germany - DAX                           :  +67.51%
  Brazil - Ibovespa                       :  +49.47%
  UK - FTSE 100                           :  +49.03%
  Hong Kong - Hang Seng                   :  +47.59%
  Australia - S&P/ASX 200                 :  +27.31%
  China - Shanghai Composite              :  +26.26%
  India - Nifty 50                        :  +25.05%
  Mexico - IPC Mexico                     :  +23.76%

--- 5y (since 2021-08-21)  | S&P 500 = +71.32%  | Beating it: 2/10 ---
  Japan - Nikkei 225                      : +140.11% <-- better
  Canada - S&P/TSX Composite              :  +78.83% <-- better
  United States - S&P 500                 :  +71.32%
  Germany - DAX                           :  +64.87%
  UK - FTSE 100  

---
## Question 3: S&P 500 Market Corrections Analysis

> Calculate the median drawdown (in %) of significant market corrections (≥5% from ATH) in S&P 500 since 1950.
>
> Also compute 25th/50th/75th percentiles for drawdown % and duration (days).


In [9]:
# Download S&P 500 daily data from 1950 to present
print("Downloading S&P 500 data from 1950...")
sp = yf.download('^GSPC', start='1950-01-01', end='2026-09-15', auto_adjust=False, progress=False)
close = sp['Close'].squeeze()
print(f"Got {len(close)} rows from {close.index.min().date()} to {close.index.max().date()}")


Got 19295 rows from 1950-01-03 to 2026-09-11


In [10]:
# Identify all-time highs and corrections between consecutive ATHs
running_max = close.cummax()
ath_mask = close == running_max
ath_dates = close.index[ath_mask]
ath_prices = close[ath_mask]
print(f"Number of all-time high days: {len(ath_dates)}")

corrections = []
for i in range(len(ath_dates) - 1):
    high_date = ath_dates[i]
    next_high_date = ath_dates[i + 1]
    high_price = float(ath_prices.iloc[i])

    # Days strictly between the two ATHs
    between = close.loc[(close.index > high_date) & (close.index < next_high_date)]
    if len(between) == 0:
        continue

    min_price = float(between.min())
    min_date = between.idxmin()
    drawdown = (high_price - min_price) / high_price * 100
    duration_days = (min_date - high_date).days           # ATH -> trough
    recovery_days = (next_high_date - high_date).days     # ATH -> new ATH (recovery)

    corrections.append({
        'high_date': high_date,
        'trough_date': min_date,
        'recovery_date': next_high_date,
        'drawdown_pct': drawdown,
        'duration_days': duration_days,
        'recovery_days': recovery_days,
    })

cdf = pd.DataFrame(corrections)
sig = cdf[cdf['drawdown_pct'] >= 5].reset_index(drop=True)
print(f"Total ATH-to-ATH events: {len(cdf)}")
print(f"Significant corrections (>=5% drawdown): {len(sig)}")


Number of all-time high days: 1537
Total ATH-to-ATH events: 681
Significant corrections (>=5% drawdown): 74


In [11]:
# Percentiles
print("=== Drawdown % percentiles ===")
for p in [25, 50, 75]:
    print(f"  {p}th: {sig['drawdown_pct'].quantile(p/100):.2f}%")

print("\n=== Duration (ATH to trough, days) percentiles ===")
for p in [25, 50, 75]:
    print(f"  {p}th: {sig['duration_days'].quantile(p/100):.0f} days")

print("\n=== Recovery time (ATH to new ATH, days) percentiles ===")
for p in [25, 50, 75]:
    print(f"  {p}th: {sig['recovery_days'].quantile(p/100):.0f} days")

print(f"\n>>> Q3 ANSWER: Median drawdown = {sig['drawdown_pct'].median():.2f}%")


=== Drawdown % percentiles ===
  25th: 6.23%
  50th: 7.99%
  75th: 14.02%

=== Duration (ATH to trough, days) percentiles ===
  25th: 22 days
  50th: 40 days
  75th: 86 days

=== Recovery time (ATH to new ATH, days) percentiles ===
  25th: 56 days
  50th: 92 days
  75th: 212 days

>>> Q3 ANSWER: Median drawdown = 7.99%


In [12]:
# Verify against homework hint: top 10 largest drawdowns
print("=== Top 10 largest corrections (verify against the homework hint) ===")
top10 = sig.sort_values('drawdown_pct', ascending=False).head(10)
for _, r in top10.iterrows():
    print(f"  {r['high_date'].date()} to {r['trough_date'].date()}: "
          f"{r['drawdown_pct']:.1f}% drawdown over {r['duration_days']} days")


=== Top 10 largest corrections (verify against the homework hint) ===
  2007-10-09 to 2009-03-09: 56.8% drawdown over 517 days
  2000-03-24 to 2002-10-09: 49.1% drawdown over 929 days
  1973-01-11 to 1974-10-03: 48.2% drawdown over 630 days
  1968-11-29 to 1970-05-26: 36.1% drawdown over 543 days
  2020-02-19 to 2020-03-23: 33.9% drawdown over 33 days
  1987-08-25 to 1987-12-04: 33.5% drawdown over 101 days
  1961-12-12 to 1962-06-26: 28.0% drawdown over 196 days
  1980-11-28 to 1982-08-12: 27.1% drawdown over 622 days
  2022-01-03 to 2022-10-12: 25.4% drawdown over 282 days
  1966-02-09 to 1966-10-07: 22.2% drawdown over 240 days


---
## Question 4: Earnings Surprise Analysis for Amazon (AMZN)

> Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.
> Then calculate the correlation between the 2-day stock return and the earnings surprise magnitude.
>
> 2-day return = Close_Day3 / Close_Day1 − 1 for three consecutive trading days where Day 2 is the earnings announcement.


In [13]:
# Get earnings dates for AMZN
ticker = 'AMZN'
t = yf.Ticker(ticker)
ed = t.get_earnings_dates(limit=100)
print(f"Earnings dates shape: {ed.shape}")
ed.head(30)


Earnings dates shape: (100, 3)


,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-10-29 16:00:00-04:00,1.95,NaN,NaN
2026-07-30 16:00:00-04:00,1.83,5.75,215.02
2026-04-29 16:00:00-04:00,1.64,2.78,69.02
2026-02-05 16:00:00-05:00,1.95,1.95,0.22
2025-10-30 16:00:00-04:00,1.56,1.95,25.20
2025-07-31 16:00:00-04:00,1.32,1.68,27.19
2025-05-01 16:00:00-04:00,1.36,1.59,16.77
2025-02-06 16:00:00-05:00,1.48,1.86,25.29
2024-10-31 16:00:00-04:00,1.14,1.43,25.22


In [14]:
# Download AMZN historical prices
hist = yf.download(ticker, start='2019-01-01', end='2026-09-15', auto_adjust=False, progress=False)
close = hist['Close'].squeeze()

# Normalize indices to naive US/Eastern dates for alignment
trading_dates = close.index.tz_localize(None).normalize()
close.index = trading_dates

ed2 = ed.copy()
ed2.index = ed2.index.tz_convert('US/Eastern').tz_localize(None).normalize()

# Homework: "25 entries starting from 2020-10-29"
ed_filtered = ed2[ed2.index >= pd.Timestamp('2020-10-29')].copy()
print(f"Earnings events from 2020-10-29: {len(ed_filtered)} (includes 1 future date with NaN EPS)")


Earnings events from 2020-10-29: 25 (includes 1 future date with NaN EPS)


In [15]:
# Compute 2-day return for each earnings event:
# Day1 = trading day BEFORE announcement, Day2 = announcement day, Day3 = day AFTER
# Return = Close_Day3 / Close_Day1 - 1
results = []
for edate, row in ed_filtered.iterrows():
    # Locate position of edate in trading days
    if edate in trading_dates:
        pos = trading_dates.get_loc(edate)
    else:
        pos_arr = np.where(trading_dates >= edate)[0]
        if len(pos_arr) == 0:
            continue
        pos = pos_arr[0]
    if pos - 1 < 0 or pos + 1 >= len(trading_dates):
        continue
    d1, d2, d3 = trading_dates[pos-1], trading_dates[pos], trading_dates[pos+1]
    p1, p3 = float(close.iloc[pos-1]), float(close.iloc[pos+1])
    results.append({
        'earnings_date': edate.date(),
        'Day1': d1.date(),
        'Day2': d2.date(),
        'Day3': d3.date(),
        '2day_return_pct': (p3 / p1 - 1) * 100,
        'Surprise_pct': row['Surprise(%)'],
        'EPS_Estimate': row['EPS Estimate'],
        'Reported_EPS': row['Reported EPS'],
    })

rdf = pd.DataFrame(results)
valid = rdf.dropna(subset=['Surprise_pct']).copy()
print(f"Events with valid Surprise data: {len(valid)}")

pos_surp = valid[valid['Surprise_pct'] > 0]
neg_surp = valid[valid['Surprise_pct'] < 0]
print(f"Positive surprises: {len(pos_surp)}  |  Negative surprises: {len(neg_surp)}")


Events with valid Surprise data: 24
Positive surprises: 20  |  Negative surprises: 4


In [16]:
# Show all events sorted by date
valid.sort_values('earnings_date')[
    ['earnings_date', 'Day1', 'Day3', '2day_return_pct', 'Surprise_pct']
].style.format({'2day_return_pct': '{:.2f}%', 'Surprise_pct': '{:.2f}%'})


,earnings_date,Day1,Day3,2day_return_pct,Surprise_pct
23,2020-10-29,2020-10-28,2020-10-30,-4.00%,64.25%
22,2021-02-02,2021-02-01,2021-02-03,-0.91%,100.10%
21,2021-04-29,2021-04-28,2021-04-30,0.26%,67.30%
20,2021-07-29,2021-07-28,2021-07-30,-8.34%,23.06%
19,2021-10-28,2021-10-27,2021-10-29,-0.59%,-31.21%
18,2022-02-03,2022-02-02,2022-02-04,4.67%,657.12%
17,2022-04-28,2022-04-27,2022-04-29,-10.05%,-190.58%
16,2022-07-28,2022-07-27,2022-07-29,11.56%,-273.30%
15,2022-10-27,2022-10-26,2022-10-28,-10.59%,35.53%
14,2023-02-02,2023-02-01,2023-02-03,-1.67%,-82.20%


In [17]:
# Median 2-day return after positive surprises
median_pos = pos_surp['2day_return_pct'].median()
median_all = valid['2day_return_pct'].median()
median_neg = neg_surp['2day_return_pct'].median() if len(neg_surp) else np.nan

# Pearson correlation (surprise % vs 2-day return %) over all valid events
corr = valid['Surprise_pct'].corr(valid['2day_return_pct'])

print(f"Median 2-day return after POSITIVE surprises : {median_pos:+.2f}%")
print(f"Median 2-day return (ALL events)             : {median_all:+.2f}%")
print(f"Median 2-day return after NEGATIVE surprises : {median_neg:+.2f}%")
print(f"Correlation (Surprise % vs 2-day return)     : {corr:.4f}")
print()
print(f">>> Q4 ANSWER: Median 2-day return after positive surprises ≈ {median_pos:.2f}%")
print(f">>>           Correlation ≈ {corr:.2f} (weak positive)")


Median 2-day return after POSITIVE surprises : +0.35%
Median 2-day return (ALL events)             : -0.17%
Median 2-day return after NEGATIVE surprises : -1.13%
Correlation (Surprise % vs 2-day return)     : 0.2191

>>> Q4 ANSWER: Median 2-day return after positive surprises ≈ 0.35%
>>>           Correlation ≈ 0.22 (weak positive)


---
## 🎯 Summary of Answers

| Question | Answer |
|---|---|
| **Q1** Year with most S&P 500 additions since 2020 | **2025** (18 additions) |
| Q1 Additional: Stocks in index > 20 years | **~224** |
| **Q2** Indices beating S&P 500 YTD (as of 2026-08-21) | **2** (Japan Nikkei 225, Canada S&P/TSX) |
| **Q3** Median drawdown of ≥5% corrections since 1950 | **≈ 8% (7.99%)** |
| **Q4** Median 2-day return after positive AMZN surprises | **≈ +0.35%** |
| Q4 Correlation (surprise % ↔ 2-day return) | **≈ 0.22** (weak positive) |
